In [1]:
pip install -q requests


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import requests
from collections import defaultdict, Counter

In [3]:
# ==========================================
# MAP ASPECT: UIT-ViSD4SA → tên aspect của mình
# ==========================================

ASPECT_MAP = {
    "BATTERY":     "battery",
    "CAMERA":      "camera",
    "SCREEN":      "display",
    "PERFORMANCE": "performance",
    "DESIGN":      "design",
    "FEATURES":    "utilities",
    "STORAGE":     "storage",
    # bỏ qua: GENERAL, PRICE, SER&ACC
}

# Keyword list hiện tại — để lọc ra từ ĐÃ CÓ
CURRENT_KEYWORDS = {
    "camera": [
        "camera", "chụp", "ảnh", "zoom", "selfie", "quay", "phim",
        "góc rộng", "telephoto", "chụp đêm", "chụp ngày", "xóa phông",
        "chân dung", "ống kính", "siêu rộng", "macro",
        "photo", "picture", "shoot", "lens", "portrait", "video",
        "footage", "nightmode", "bokeh", "ultrawide", "snapshot",
    ],
    "battery": [
        "pin", "sạc", "hao pin", "nạp điện", "hao nhanh", "sạc nhanh",
        "cạn pin", "trâu pin", "pin trâu", "thời lượng pin", "sạc không dây",
        "battery", "charge", "charging", "drain", "fast charge",
        "wireless charge", "battery life", "power bank",
    ],
    "display": [
        "màn hình", "màn", "độ sáng", "tần số quét", "ám vàng",
        "burn-in", "tấm nền", "độ phân giải", "notch", "đục lỗ",
        "oled", "amoled", "lcd",
        "screen", "display", "nit", "refresh rate", "resolution",
        "panel", "brightness", "pwm", "punchhole",
    ],
    "performance": [
        "hiệu năng", "lag", "mượt", "chậm", "giật", "nóng máy",
        "đơ", "nhanh", "hiệu suất", "tản nhiệt", "chip",
        "vi xử lý", "xử lý", "ram",
        "snapdragon", "dimensity", "cpu", "gpu", "processor",
        "benchmark", "smooth", "heating", "throttle", "performance",
        "helio", "gaming", "freeze",
    ],
    "design": [
        "thiết kế", "mỏng", "nhẹ", "nặng", "màu sắc", "vỏ máy",
        "kính", "nhôm", "titan", "nhựa", "sang trọng", "ngoại hình",
        "design", "build quality", "color", "weight", "thin", "premium",
        "glass", "aluminum", "titanium", "finish", "aesthetic",
    ],
    "storage": [
        "bộ nhớ", "lưu trữ", "dung lượng", "rom", "thẻ nhớ",
        "bộ nhớ trong", "bộ nhớ ngoài", "đầy bộ nhớ",
        "storage", "sd card", "microsd", "internal storage", "capacity",
    ],
    "connectivity": [
        "wifi", "sóng", "kết nối", "sim", "mạng", "hotspot", "jack tai nghe",
        "bluetooth", "nfc", "network", "signal", "lte", "type-c",
        "5g", "4g", "3g",
    ],
    "utilities": [
        "tính năng", "vân tay", "chống nước", "nhận diện khuôn mặt",
        "loa ngoài", "âm thanh", "stereo", "always on display",
        "face id", "fingerprint", "ip68", "ip67", "waterproof",
        "speaker", "audio", "dolby", "haptic", "aod",
    ],
}

# Stopwords tiếng Việt phổ biến — lọc khỏi kết quả
VI_STOPWORDS = {
    "là", "và", "của", "có", "được", "không", "thì", "để", "với", "trong",
    "này", "đó", "nhưng", "rất", "quá", "lắm", "cũng", "đã", "bị", "cho",
    "nên", "mà", "hay", "như", "cái", "con", "máy", "điện", "thoại", "sản",
    "phẩm", "mua", "dùng", "dùng", "dùng", "tốt", "đẹp", "xấu", "ổn",
    "ok", "oke", "okay", "nha", "nhe", "nhé", "ạ", "ơi", "vậy", "thế",
    "bị", "khi", "lúc", "lại", "ra", "vào", "lên", "xuống", "từ", "hơn",
    "nhất", "thật", "thực", "chất", "giá", "tiền", "hàng", "shop", "ship",
    "mình", "tôi", "tui", "bạn", "bn", "ae", "các", "mấy", "tất", "cả",
    "đều", "chưa", "rồi", "vẫn", "còn", "chỉ", "cần", "thêm", "bỏ",
    "ngoài", "nếu", "thì", "so", "so", "the", "a", "an", "is", "it",
    "to", "of", "in", "and", "or", "but", "for", "on", "at", "by",
}

In [4]:
# ==========================================
# TẢI VÀ PARSE DỮ LIỆU
# ==========================================

URL_TRAIN = "https://raw.githubusercontent.com/kimkim00/UIT-ViSD4SA/main/data/train.jsonl"
URL_DEV   = "https://raw.githubusercontent.com/kimkim00/UIT-ViSD4SA/main/data/dev.jsonl"

def load_jsonl(url):
    print(f"Đang tải: {url}")
    r = requests.get(url)
    r.raise_for_status()
    lines = [json.loads(l) for l in r.text.strip().splitlines() if l.strip()]
    print(f"  → {len(lines):,} samples")
    return lines

data = load_jsonl(URL_TRAIN) + load_jsonl(URL_DEV)
print(f"Tổng: {len(data):,} samples")

Đang tải: https://raw.githubusercontent.com/kimkim00/UIT-ViSD4SA/main/data/train.jsonl
  → 7,785 samples
Đang tải: https://raw.githubusercontent.com/kimkim00/UIT-ViSD4SA/main/data/dev.jsonl
  → 1,112 samples
Tổng: 8,897 samples


In [5]:
# ==========================================
# EXTRACT SPANS VÀ ĐẾM N-GRAM THEO ASPECT
# ==========================================
# Lấy cả 1-gram, 2-gram, 3-gram, 4-gram
# vì tiếng Việt hay dùng cụm từ 2-4 từ

WORD_RE = re.compile(
    r'[\wàáâãèéêìíòóôõùúýăđơưạảấầẩẫậắằẳẵặẹẻẽếềểễệỉịọỏốồổỗộớờởỡợụủứừửữựỳỵỷỹ]+'
)

def extract_ngrams(text, max_n=4):
    tokens = [t for t in WORD_RE.findall(text.lower()) if len(t) >= 2]
    ngrams = []
    for n in range(1, max_n + 1):
        for i in range(len(tokens) - n + 1):
            ngrams.append(" ".join(tokens[i:i+n]))
    return ngrams

# Đếm n-gram trong span text của mỗi aspect
aspect_ngram_counts = defaultdict(Counter)
aspect_span_counts  = defaultdict(int)

for sample in data:
    text = sample["text"]
    for start, end, label in sample["labels"]:
        aspect_raw = label.split("#")[0]
        aspect     = ASPECT_MAP.get(aspect_raw)
        if not aspect:
            continue
        span_text = text[start:end]
        ngrams    = extract_ngrams(span_text)
        aspect_ngram_counts[aspect].update(ngrams)
        aspect_span_counts[aspect] += 1

print("Số spans đã extract:")
for aspect, count in sorted(aspect_span_counts.items(), key=lambda x: -x[1]):
    print(f"  {aspect:<15} {count:>5,} spans")

Số spans đã extract:
  performance     5,538 spans
  battery         4,516 spans
  utilities       3,171 spans
  camera          2,500 spans
  design          1,731 spans
  display         1,106 spans
  storage           111 spans


In [6]:
# ==========================================
# IN TOP CANDIDATE N-GRAM CHO TỪNG ASPECT
# ==========================================
# Lọc: chưa có trong keyword list + không bắt/kết bằng stopword

existing_kw = {kw.lower() for kws in CURRENT_KEYWORDS.values() for kw in kws}

# Tổng tần suất mỗi n-gram trên toàn bộ aspects
total_counts = Counter()
for counts in aspect_ngram_counts.values():
    total_counts.update(counts)

def is_valid_ngram(ngram):
    parts = ngram.split()
    if parts[0] in VI_STOPWORDS or parts[-1] in VI_STOPWORDS:
        return False
    if all(p.isdigit() for p in parts):
        return False
    return True

for aspect in ASPECT_MAP.values():
    if aspect not in aspect_ngram_counts:
        continue

    counts = aspect_ngram_counts[aspect]

    scored = []
    for ngram, freq in counts.items():
        if ngram in existing_kw:
            continue
        if not is_valid_ngram(ngram):
            continue
        if freq < 2:
            continue
        specificity = freq / total_counts[ngram]
        score = freq * specificity
        scored.append((ngram, freq, round(specificity, 2), round(score, 1)))

    scored.sort(key=lambda x: -x[3])

    unigrams = [(g, f, s, sc) for g, f, s, sc in scored if len(g.split()) == 1]
    bigrams  = [(g, f, s, sc) for g, f, s, sc in scored if len(g.split()) == 2]
    trigrams = [(g, f, s, sc) for g, f, s, sc in scored if len(g.split()) >= 3]

    print(f"\n{'='*60}")
    print(f"  {aspect.upper()}")
    print(f"{'='*60}")
    print(f"  {'N-gram':<28} {'Freq':>5}  {'Spec':>5}  {'Score':>6}")
    print(f"  {'-'*28} {'-'*5}  {'-'*5}  {'-'*6}")

    print(f"\n  -- 1-gram --")
    for g, f, s, sc in unigrams[:15]:
        print(f"  {g:<28} {f:>5}  {s:>5.2f}  {sc:>6.1f}")

    print(f"\n  -- 2-gram --")
    for g, f, s, sc in bigrams[:20]:
        print(f"  {g:<28} {f:>5}  {s:>5.2f}  {sc:>6.1f}")

    print(f"\n  -- 3-gram trở lên --")
    for g, f, s, sc in trigrams[:15]:
        print(f"  {g:<28} {f:>5}  {s:>5.2f}  {sc:>6.1f}")


  BATTERY
  N-gram                        Freq   Spec   Score
  ---------------------------- -----  -----  ------

  -- 1-gram --
  trâu                          1359   0.99  1346.1
  ngày                           494   0.79   391.1
  tụt                            404   0.94   380.5
  hết                            445   0.74   328.9
  đầy                            270   0.91   244.6
  tuột                           224   0.92   205.6
  tiếng                          245   0.75   184.7
  xài                            274   0.53   144.4
  lâu                            309   0.45   138.8
  hao                            130   0.91   118.2
  bin                             97   0.94    91.3
  mới                            234   0.39    91.3
  mau                            114   0.79    90.2
  sài                            162   0.53    86.0
  đến                            142   0.58    82.0

  -- 2-gram --
  sạc pin                        181   0.94   170.6
  hết pin            